# 🦎 Lizard Species Classification – Kaggle Challenge

**Team:** _[vul hier jullie teamnaam in]_  
**Members:** _[naam 1]_, _[naam 2]_, _[naam 3]_  
**Date:** _[datum]_

---

## Overzicht

In dit notebook klassificeren we 7 soorten hagedissen voor de Kaggle challenge.

| Label | Soort |
|-------|-------|
| 0 | Black_spiny_tailed_iguana |
| 1 | Brown_anole |
| 2 | Cuban_knight_anole |
| 3 | Desert_iguana |
| 4 | Green_anole |
| 5 | Green_iguana |
| 6 | Lesser_Antillean_iguana |

**Dataset:**
- 1334 trainingsafbeeldingen in `train/` (7 submappen, 1 per soort)
- 326 testafbeeldingen in `test/` (genummerd: `1.jpg` t/m `326.jpg`)

**Stappen:**
1. EDA – Exploratory Data Analysis
2. Eigen CNN model (baseline)
3. Transfer Learning met EfficientNetV2S + Fine-tuning
4. Data Augmentation
5. Trainingscurves
6. Confusion Matrix & evaluatie
7. Kaggle Submission

> ⚠️ **Tip:** Voer dit notebook uit op Google Colab met GPU-runtime voor snellere training.

---
## 0. Setup & Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as eff_preprocess
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, classification_report
from PIL import Image

print('TensorFlow versie:', tf.__version__)
print('GPU beschikbaar:', tf.config.list_physical_devices('GPU'))

In [ ]:
# Paden
TRAIN_DIR = 'train'   # bevat 7 submappen
TEST_DIR  = 'test'    # bevat 1.jpg t/m 326.jpg

# Hyperparameters
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
VAL_SPLIT  = 0.2
SEED       = 42
AUTOTUNE   = tf.data.AUTOTUNE

# Klassen (alfabetisch = dezelfde volgorde als Keras inference)
CLASS_NAMES = [
    'Black_spiny_tailed_iguana',  # 0
    'Brown_anole',                # 1
    'Cuban_knight_anole',         # 2
    'Desert_iguana',              # 3
    'Green_anole',                # 4
    'Green_iguana',               # 5
    'Lesser_Antillean_iguana'     # 6
]
NUM_CLASSES = len(CLASS_NAMES)
print(f'{NUM_CLASSES} klassen geladen')

---
## 1. EDA – Exploratory Data Analysis

In [ ]:
# Aantal afbeeldingen per klasse
train_df = pd.read_csv('train.csv')
train_df['class_name'] = train_df['label'].map(dict(enumerate(CLASS_NAMES)))
counts = train_df['class_name'].value_counts().reindex(CLASS_NAMES)

print(f'Totaal trainingsafbeeldingen: {len(train_df)}')
print(counts)

In [ ]:
# Klassenbalans: bar + pie
colors = plt.cm.Set2.colors
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(NUM_CLASSES), counts.values, color=colors, edgecolor='black')
axes[0].set_xticks(range(NUM_CLASSES))
axes[0].set_xticklabels([c.replace('_', '\n') for c in CLASS_NAMES], fontsize=8)
axes[0].set_title('Aantal afbeeldingen per klasse')
axes[0].set_ylabel('Aantal')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1, str(v), ha='center', fontsize=9)

axes[1].pie(counts.values, labels=[c.replace('_', ' ') for c in CLASS_NAMES],
            colors=colors, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Verdeling (%)')

plt.suptitle('EDA – Klassenbalans trainingsset', fontsize=14)
plt.tight_layout()
plt.show()

if counts.max() / counts.min() > 2:
    print('Ongebalanceerd! Overweeg class_weight.')
else:
    print('Dataset is redelijk gebalanceerd.')

In [ ]:
# Voorbeeldafbeeldingen per soort
fig, axes = plt.subplots(NUM_CLASSES, 4, figsize=(14, NUM_CLASSES * 3))
for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(TRAIN_DIR, cls)
    imgs = sorted(os.listdir(cls_path))[:4]
    for col, img_name in enumerate(imgs):
        img = Image.open(os.path.join(cls_path, img_name)).resize(IMAGE_SIZE)
        axes[row, col].imshow(img)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_title(cls.replace('_', ' '), fontsize=8, loc='left')
plt.suptitle('EDA – Voorbeeldafbeeldingen per soort', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Afbeeldingsgroottes (steekproef)
widths, heights = [], []
for cls in CLASS_NAMES:
    for fn in sorted(os.listdir(os.path.join(TRAIN_DIR, cls)))[:15]:
        try:
            w, h = Image.open(os.path.join(TRAIN_DIR, cls, fn)).size
            widths.append(w); heights.append(h)
        except: pass

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(widths,  bins=20, color='cornflowerblue', edgecolor='black')
axes[0].set_title(f'Breedte (gem. {np.mean(widths):.0f} px)')
axes[1].hist(heights, bins=20, color='salmon',         edgecolor='black')
axes[1].set_title(f'Hoogte (gem. {np.mean(heights):.0f} px)')
plt.suptitle('EDA – Afbeeldingsformaten', fontsize=13)
plt.tight_layout()
plt.show()
print(f'We resizen alles naar {IMAGE_SIZE[0]}x{IMAGE_SIZE[1]} px.')

---
## 2. Data Laden & Data Augmentation

In [ ]:
# Trainings- en validatieset vanuit mappenstructuur
train_ds = image_dataset_from_directory(
    TRAIN_DIR, labels='inferred', label_mode='int',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT, subset='training', seed=SEED
)
val_ds = image_dataset_from_directory(
    TRAIN_DIR, labels='inferred', label_mode='int',
    image_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT, subset='validation', seed=SEED
)

# Keras leest klassen alfabetisch -> zelfde als CLASS_NAMES
assert train_ds.class_names == CLASS_NAMES, 'Klassenordening klopt niet!'
print('Klassenordening OK:', train_ds.class_names)

In [ ]:
# Data Augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.10),
    layers.RandomContrast(0.10),
    layers.RandomBrightness(0.10),
], name='data_augmentation')

# Visualisatie van augmentatie
for images, labels in train_ds.take(1):
    sample_img = images[0]
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes[0,0].imshow(sample_img.numpy().astype('uint8'))
    axes[0,0].set_title('Origineel'); axes[0,0].axis('off')
    for i, ax in enumerate(axes.flatten()[1:]):
        aug = data_augmentation(tf.expand_dims(sample_img, 0))[0]
        ax.imshow(aug.numpy().astype('uint8'))
        ax.set_title(f'Aug {i+1}', fontsize=8); ax.axis('off')
    plt.suptitle('Data Augmentation – 9 variaties', fontsize=13)
    plt.tight_layout(); plt.show()

In [ ]:
# Prefetch voor snelheid
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

---
## 3. Eigen CNN Model (Baseline)

In [ ]:
own_model = models.Sequential([
    layers.Input(shape=(*IMAGE_SIZE, 3)),
    layers.Rescaling(1./255),
    data_augmentation,

    layers.Conv2D(32,  (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)), layers.Dropout(0.2),

    layers.Conv2D(64,  (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)), layers.Dropout(0.2),

    layers.Conv2D(128, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)), layers.Dropout(0.3),

    layers.Conv2D(256, (3,3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)), layers.Dropout(0.3),

    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='eigen_cnn')

own_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
own_model.summary()

In [ ]:
history_own = own_model.fit(
    train_ds, validation_data=val_ds, epochs=20
)

---
## 4. Trainingscurves

In [ ]:
def plot_training(history, title='Trainingscurves'):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    ax1.plot(history.history['loss'],     label='Training',   color='steelblue')
    ax1.plot(history.history['val_loss'], label='Validatie',  color='orange')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(history.history['accuracy'],     label='Training',  color='steelblue')
    ax2.plot(history.history['val_accuracy'], label='Validatie', color='orange')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.suptitle(title, fontsize=14)
    plt.tight_layout(); plt.show()
    print(f'Beste validatie-accuracy: {max(history.history["val_accuracy"]):.4f}')

plot_training(history_own, 'Eigen CNN – Trainingscurves')

---
## 5. Transfer Learning – EfficientNetV2S

**Fase 1:** base bevroren, enkel eigen hoofd trainen  
**Fase 2:** fine-tuning – laatste lagen ontdooien met lage learning rate

In [ ]:
# EfficientNet preprocessing (geen Rescaling nodig)
def preprocess_eff(image, label):
    return eff_preprocess(tf.cast(image, tf.float32)), label

train_ds_eff = train_ds.map(preprocess_eff, num_parallel_calls=AUTOTUNE)
val_ds_eff   = val_ds.map(preprocess_eff,   num_parallel_calls=AUTOTUNE)

# Base model laden
base_model = EfficientNetV2S(
    input_shape=(*IMAGE_SIZE, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
print(f'EfficientNetV2S: {len(base_model.layers)} lagen, bevroren')

In [ ]:
# Eigen hoofd bouwen
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3))
x = data_augmentation(inputs)
x = base_model(x, training=False)
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

tl_model = tf.keras.Model(inputs, outputs, name='efficientnet_transfer')
tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
tl_model.summary()

In [ ]:
# Fase 1: hoofd trainen
print('--- Fase 1: hoofd trainen (base bevroren) ---')
history_tl = tl_model.fit(
    train_ds_eff, validation_data=val_ds_eff, epochs=10
)
plot_training(history_tl, 'Transfer Learning – Fase 1 (bevroren base)')

In [ ]:
# Fase 2: fine-tuning
base_model.trainable = True
fine_tune_at = 200
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f'Fine-tuning: {sum(l.trainable for l in base_model.layers)} van {len(base_model.layers)} lagen trainbaar')

tl_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),  # lagere LR voor fine-tuning
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('--- Fase 2: fine-tuning ---')
history_ft = tl_model.fit(
    train_ds_eff, validation_data=val_ds_eff, epochs=10
)
plot_training(history_ft, 'Transfer Learning – Fase 2 (fine-tuning)')

---
## 6. Evaluatie & Confusion Matrix

In [ ]:
def evaluate_model(model, dataset, title='Model'):
    all_labels, all_preds = [], []
    for images, labels in dataset:
        preds = model.predict(images, verbose=0)
        all_preds.extend(np.argmax(preds, axis=1))
        all_labels.extend(labels.numpy())

    acc = accuracy_score(all_labels, all_preds)
    cm  = confusion_matrix(all_labels, all_preds)
    short = [c.replace('_', ' ') for c in CLASS_NAMES]

    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=short).plot(
        cmap=plt.cm.Blues, ax=ax, colorbar=False
    )
    ax.set_title(f'{title}  |  Validatie accuracy = {acc:.4f}', fontsize=12)
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout(); plt.show()

    abbrev = [c.split('_')[0][:8] for c in CLASS_NAMES]
    print(f'\nClassification Report – {title}')
    print(classification_report(all_labels, all_preds, target_names=abbrev))
    return acc

print('='*55)
acc_own = evaluate_model(own_model, val_ds,     title='Eigen CNN')
print('='*55)
acc_tl  = evaluate_model(tl_model,  val_ds_eff, title='EfficientNetV2S fine-tuned')

print('\nSAMENVATTING')
print(f'  Eigen CNN              : {acc_own:.4f}')
print(f'  Transfer Learning      : {acc_tl:.4f}')
print(f'  Beste model            : {"EfficientNetV2S" if acc_tl > acc_own else "Eigen CNN"}')

---
## 7. Kaggle Submission

Testset: 326 afbeeldingen, genummerd `1.jpg` t/m `326.jpg`.  
Submission formaat: `id,label` (zie `sample_submission.csv`).

In [ ]:
test_csv = pd.read_csv('test.csv')
test_ids = test_csv['id'].tolist()
print(f'Testset: {len(test_ids)} afbeeldingen (id {test_ids[0]} t/m {test_ids[-1]})')

In [ ]:
# Testafbeeldingen laden in volgorde van de IDs
X_test = []
for img_id in test_ids:
    img = tf.keras.utils.load_img(
        os.path.join(TEST_DIR, f'{img_id}.jpg'),
        target_size=IMAGE_SIZE
    )
    X_test.append(tf.keras.utils.img_to_array(img))

X_test = np.array(X_test)
print(f'Test array: {X_test.shape}')

In [ ]:
# Preprocessing + voorspellingen
X_test_eff  = eff_preprocess(X_test.astype('float32'))
predictions = tl_model.predict(X_test_eff, batch_size=BATCH_SIZE, verbose=1)
pred_labels = np.argmax(predictions, axis=1)

print('Verdeling voorspellingen:')
for cls_id, cnt in zip(*np.unique(pred_labels, return_counts=True)):
    print(f'  {cls_id} – {CLASS_NAMES[cls_id]}: {cnt}')

In [ ]:
# Submission aanmaken
submission = pd.DataFrame({'id': test_ids, 'label': pred_labels})
submission.to_csv('submission.csv', index=False)
print('submission.csv opgeslagen!')
submission.head(10)

In [ ]:
# Verdeling visualiseren
pred_counts = pd.Series(pred_labels).map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES)
plt.figure(figsize=(10, 4))
plt.bar(range(NUM_CLASSES), pred_counts.values, color=plt.cm.Set2.colors, edgecolor='black')
plt.xticks(range(NUM_CLASSES), [c.replace('_', '\n') for c in CLASS_NAMES], fontsize=8)
plt.title('Verdeling voorspellingen – testset (326 afbeeldingen)')
plt.ylabel('Aantal')
for i, v in enumerate(pred_counts.values):
    plt.text(i, v + 0.2, str(v), ha='center', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# Model opslaan
tl_model.save('lizard_model.keras')
print('Model opgeslagen als lizard_model.keras')

---
## 8. GenAI Policy

**Welk AI-tool hebben we gebruikt?**  
_[bv. Claude, ChatGPT, ...]_

**Waarvoor hebben we het gebruikt?**  
_[bv. structuur van het notebook, uitleg van functies, debugging]_

**Welke prompts hebben we gebruikt?**  
_[voeg hier de prompts in]_

**Beperkingen en onzekerheden:**  
_[bv. we hebben alle code zelf begrepen en getest voor indiening]_